# E3 Fashion-MNIST: LSTM vs GRU

Colab-ready training and test evaluation for the E3 recurrent-image-classification pipeline. This notebook reuses the project modules, prefers an available GPU automatically, requires an existing Fashion-MNIST cache in Google Drive, and writes experiment artifacts back to Drive. It never downloads datasets.


## 1. Environment setup and configurable paths


In [ ]:
from pathlib import Path
import inspect
import json
import os
import subprocess
import sys
import time

# All project, source-data, and output locations can be changed here.
REPOSITORY_URL = "https://github.com/hoctran19-cyber/deep-learning-computer-vision-co5085.git"
BRANCH = "ex_3"
REPO_DIR = Path("/content/deep-learning-computer-vision-co5085")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive"
# A direct override is optional; when None, the notebook searches E1 locations in Drive.
FASHION_MNIST_DATA_DIR = None  # e.g. DRIVE_ROOT / "deep-learning-computer-vision-co5085/E1/data"
E1_DATA_SEARCH_ROOTS = [
    DRIVE_ROOT / "deep-learning-computer-vision-co5085/E1/data",
    DRIVE_ROOT / "E1/data",
]
OUTPUT_DIR = DRIVE_ROOT / "E3_experiments/fashion_mnist_lstm_gru"
INSTALL_DEPENDENCIES = True
SEED = 42
VALIDATION_SPLIT = 0.10
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3
NUM_WORKERS = 2

IN_COLAB = Path("/content").exists() and "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("This notebook is intended for Google Colab. Open it in Colab and run top-to-bottom.")
print(f"Python: {sys.version.split()[0]} | Colab: {IN_COLAB}")


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount(str(DRIVE_MOUNT), force_remount=False)
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f"Google Drive MyDrive was not found at {DRIVE_ROOT}")
print(f"Drive mounted: {DRIVE_ROOT}")


## 3. Clone or update branch `ex_3` and install dependencies


In [ ]:
if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository; change REPO_DIR in the setup cell.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(f"Repository ready: {REPO_DIR} (branch {BRANCH})")

if INSTALL_DEPENDENCIES:
    requirements = REPO_DIR / "requirements.txt"
    if not requirements.is_file():
        raise FileNotFoundError(f"Dependency file not found: {requirements}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)

# Import the E3 package from the checked-out project root.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Dependencies are ready.")


## 4. Find and verify the existing E1 Fashion-MNIST dataset

Only an already-processed torchvision Fashion-MNIST cache is accepted. If it is absent, execution stops; no download or fallback dataset is permitted. Adjust `FASHION_MNIST_DATA_DIR` or `E1_DATA_SEARCH_ROOTS` in the setup cell to match your Drive layout.


In [ ]:
def is_fashion_mnist_cache(data_root):
    processed = Path(data_root) / "FashionMNIST" / "processed"
    return processed.is_dir() and (processed / "training.pt").is_file() and (processed / "test.pt").is_file()

def discover_e1_data(search_roots):
    # Check explicitly configured roots first, then a bounded-depth Drive walk for E1/data.
    candidates = [Path(p) for p in search_roots]
    seen = {str(p.resolve()) for p in candidates if p.exists()}
    max_depth = 5
    for current, dirs, _ in os.walk(DRIVE_ROOT):
        current_path = Path(current)
        depth = len(current_path.relative_to(DRIVE_ROOT).parts)
        dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {".Trash", ".ipynb_checkpoints"}] if depth < max_depth else []
        if current_path.name == "E1":
            candidate = current_path / "data"
            key = str(candidate.resolve())
            if key not in seen:
                candidates.append(candidate)
                seen.add(key)
    for candidate in candidates:
        if is_fashion_mnist_cache(candidate):
            return candidate
    checked = "\n".join(f"  - {p}" for p in candidates)
    raise FileNotFoundError(
        "No existing processed Fashion-MNIST cache was found in the configured E1 Drive locations.\n"
        "This notebook will not download data. Copy the E1 FashionMNIST cache into E1/data/FashionMNIST/processed/ or update the search paths.\n"
        f"Checked:\n{checked}"
    )

DATA_DIR = Path(FASHION_MNIST_DATA_DIR) if FASHION_MNIST_DATA_DIR else discover_e1_data(E1_DATA_SEARCH_ROOTS)
if not is_fashion_mnist_cache(DATA_DIR):
    raise FileNotFoundError(f"Fashion-MNIST cache is incomplete at {DATA_DIR}; refusing to download.")
print(f"Using existing E1 Fashion-MNIST cache: {DATA_DIR}")
print(f"Processed files: {(DATA_DIR / 'FashionMNIST/processed/training.pt').name}, {(DATA_DIR / 'FashionMNIST/processed/test.pt').name}")


## 5. Load data and verify train/validation/test sizes


In [ ]:
import torch
from E3.src.datasets import load_datasets, create_dataloaders
from E3.src.models import get_model
from E3.src.train import train_model
from E3.src.evaluate import evaluate_checkpoint
from E3.src.utils import get_device, save_history, save_experiment_results

def call_project_api(function, values, aliases):
    """Pass supported keyword arguments while accommodating small API naming differences."""
    signature = inspect.signature(function)
    parameters = signature.parameters
    accepts_extra = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in parameters.values())
    kwargs = {}
    for semantic_name, value in values.items():
        candidates = aliases.get(semantic_name, (semantic_name,))
        matching = next((name for name in candidates if name in parameters), None)
        if matching is not None:
            kwargs[matching] = value
        elif accepts_extra:
            kwargs[candidates[0]] = value
    missing = [
        p.name for p in parameters.values()
        if p.default is inspect.Parameter.empty
        and p.kind in (inspect.Parameter.POSITIONAL_ONLY, inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)
        and p.name not in kwargs
    ]
    if missing:
        raise TypeError(f"Cannot call {function.__module__}.{function.__name__}{signature}: no value mapped for required parameter(s) {missing}.")
    return function(**kwargs)

dataset_values = {
    "data_dir": str(DATA_DIR), "download": False, "seed": SEED,
    "validation_split": VALIDATION_SPLIT, "dataset_name": "fashion_mnist",
}
dataset_aliases = {
    "data_dir": ("data_dir", "root", "dataset_dir"),
    "download": ("download",), "seed": ("seed",),
    "validation_split": ("validation_split", "val_split", "validation_ratio"),
    "dataset_name": ("dataset_name", "name"),
}
datasets = call_project_api(load_datasets, dataset_values, dataset_aliases)
if isinstance(datasets, dict):
    train_dataset = datasets.get("train", datasets.get("train_dataset"))
    val_dataset = datasets.get("validation", datasets.get("val", datasets.get("val_dataset")))
    test_dataset = datasets.get("test", datasets.get("test_dataset"))
else:
    train_dataset, val_dataset, test_dataset = datasets
if any(dataset is None for dataset in (train_dataset, val_dataset, test_dataset)):
    raise RuntimeError("load_datasets must provide train, validation, and test datasets.")

loader_values = {
    "train_dataset": train_dataset, "val_dataset": val_dataset, "test_dataset": test_dataset,
    "datasets": datasets, "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS, "seed": SEED,
}
loader_aliases = {
    "train_dataset": ("train_dataset", "train_data", "train"),
    "val_dataset": ("val_dataset", "validation_dataset", "val_data", "validation_data", "val"),
    "test_dataset": ("test_dataset", "test_data", "test"),
    "datasets": ("datasets",), "batch_size": ("batch_size",),
    "num_workers": ("num_workers",), "seed": ("seed",),
}
loaders = call_project_api(create_dataloaders, loader_values, loader_aliases)
if isinstance(loaders, dict):
    train_loader = loaders.get("train", loaders.get("train_loader"))
    val_loader = loaders.get("validation", loaders.get("val", loaders.get("val_loader")))
    test_loader = loaders.get("test", loaders.get("test_loader"))
else:
    train_loader, val_loader, test_loader = loaders
sizes = {"train": len(train_loader.dataset), "validation": len(val_loader.dataset), "test": len(test_loader.dataset)}
print("Dataset sizes:")
for split, count in sizes.items():
    print(f"  {split:>10}: {count:,}")
assert sizes["train"] + sizes["validation"] == 60_000, f"Expected 60,000 training examples total, got {sizes['train'] + sizes['validation']:,}"
assert sizes["test"] == 10_000, f"Expected 10,000 test examples, got {sizes['test']:,}"
assert abs(sizes["validation"] / 60_000 - VALIDATION_SPLIT) < 0.002, "Validation split differs from configured ratio."
print("Verified Fashion-MNIST train/validation/test counts.")


## 6. Select and report the compute device


In [ ]:
DEVICE = get_device()
print(f"Selected device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime: {torch.version.cuda}")
else:
    print("No CUDA GPU detected; the project-selected device will be used.")
if hasattr(torch.backends, "mps"):
    print(f"Apple MPS available: {torch.backends.mps.is_available()}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Experiment artifacts will be saved to: {OUTPUT_DIR}")


## 7. Train LSTM and GRU, then evaluate their best checkpoints

Both models use the same verified data splits and training settings. Checkpoints, JSON history, and experiment metrics are written under `OUTPUT_DIR` on Drive.


In [ ]:
MODEL_CONFIG = {"input_size": 28, "hidden_size": 128, "num_layers": 2, "dropout": 0.2, "num_classes": 10}
TRAIN_CONFIG = {"seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE}
train_aliases = {
    "model": ("model", "network"), "model_name": ("model_name", "name"),
    "train_loader": ("train_loader", "train_dataloader"),
    "val_loader": ("val_loader", "validation_loader", "val_dataloader"),
    "device": ("device",), "config": ("config", "training_config"),
    "checkpoint_path": ("checkpoint_path", "best_checkpoint_path", "save_path", "checkpoint_file"),
    "epochs": ("epochs", "num_epochs"),
    "learning_rate": ("learning_rate", "lr"),
}
eval_aliases = {
    "model": ("model", "network"),
    "checkpoint_path": ("checkpoint_path", "checkpoint", "path", "best_checkpoint_path"),
    "test_loader": ("test_loader", "dataloader", "data_loader", "loader"),
    "device": ("device",),
}
history_aliases = {"history": ("history",), "path": ("path", "file_path", "output_path")}
results_aliases = {
    "results": ("results", "experiment_results", "metrics"),
    "output_dir": ("output_dir", "results_dir", "directory"),
    "output_path": ("output_path", "file_path", "path"),
    "filename": ("filename", "name"),
}

experiment_results = {}
comparison_rows = []
for model_name in ("lstm", "gru"):
    print(f"\n{'=' * 18} Training {model_name.upper()} {'=' * 18}")
    model = get_model(model_name, MODEL_CONFIG).to(DEVICE)
    checkpoint_path = OUTPUT_DIR / f"{model_name}_best.pt"
    history_path = OUTPUT_DIR / f"{model_name}_history.json"
    train_values = {
        "model": model, "model_name": model_name, "train_loader": train_loader,
        "val_loader": val_loader, "device": DEVICE, "config": TRAIN_CONFIG,
        "checkpoint_path": str(checkpoint_path), "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
    }
    started = time.perf_counter()
    training_output = call_project_api(train_model, train_values, train_aliases)
    elapsed = time.perf_counter() - started
    if isinstance(training_output, tuple):
        history = training_output[0]
        if len(training_output) > 1 and isinstance(training_output[1], (str, Path)):
            checkpoint_path = Path(training_output[1])
    elif isinstance(training_output, dict) and "history" in training_output:
        history = training_output["history"]
        checkpoint_path = Path(training_output.get("checkpoint_path", checkpoint_path))
    else:
        history = training_output
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Training did not produce the expected best checkpoint: {checkpoint_path}")
    call_project_api(save_history, {"history": history, "path": str(history_path)}, history_aliases)
    evaluation = call_project_api(
        evaluate_checkpoint,
        {"model": model, "checkpoint_path": str(checkpoint_path), "test_loader": test_loader, "device": DEVICE},
        eval_aliases,
    )
    if not isinstance(evaluation, dict):
        evaluation = {"test_result": evaluation}
    experiment_results[model_name] = {
        "model": model_name, "checkpoint_path": str(checkpoint_path),
        "history_path": str(history_path), "training_seconds": elapsed,
        "train_size": sizes["train"], "validation_size": sizes["validation"],
        "test_size": sizes["test"], "evaluation": evaluation,
    }
    metrics_path = OUTPUT_DIR / f"{model_name}_metrics.json"
    metrics_path.write_text(json.dumps(experiment_results[model_name], indent=2, default=str) + "\n", encoding="utf-8")
    call_project_api(
        save_experiment_results,
        {"results": experiment_results[model_name], "output_dir": str(OUTPUT_DIR), "output_path": str(OUTPUT_DIR / f"{model_name}_results.json"), "filename": f"{model_name}_results.json"},
        results_aliases,
    )
    accuracy = next((evaluation[k] for k in ("test_accuracy", "accuracy", "test_acc") if k in evaluation), None)
    comparison_rows.append((model_name.upper(), accuracy, elapsed, str(checkpoint_path)))
    print(f"Best checkpoint: {checkpoint_path}")
    print(f"Test metrics: {evaluation}")

# Save a combined, durable copy in Drive as well as each model's individual artifacts.
combined_path = OUTPUT_DIR / "experiment_results.json"
combined_path.write_text(json.dumps(experiment_results, indent=2, default=str) + "\n", encoding="utf-8")
call_project_api(
    save_experiment_results,
    {"results": experiment_results, "output_dir": str(OUTPUT_DIR), "output_path": str(combined_path), "filename": "experiment_results.json"},
    results_aliases,
)
print(f"\nAll checkpoints and histories saved in {OUTPUT_DIR}")


## 8. LSTM/GRU comparison


In [ ]:
from IPython.display import display, HTML

def show_comparison(rows):
    headings = ("Model", "Test accuracy", "Training time (s)", "Best checkpoint")
    body = []
    for model_name, accuracy, elapsed, checkpoint in rows:
        accuracy_text = f"{float(accuracy):.4f}" if accuracy is not None else "See test metrics above"
        body.append(f"<tr><td>{model_name}</td><td>{accuracy_text}</td><td>{elapsed:.1f}</td><td><code>{checkpoint}</code></td></tr>")
    table = "<table><thead><tr>" + "".join(f"<th>{h}</th>" for h in headings) + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"
    display(HTML(table))

show_comparison(comparison_rows)
print(f"Full per-model metrics and combined results: {OUTPUT_DIR}")
